# Week 5 Activity — Break My Schema

## What's this all about?

Have you ever filled out a form online that wouldn't take your name? Maybe it had a hyphen or an apostrophe, or it was one word instead of two, or the form wanted a ZIP code and you live somewhere that doesn't have them. Somebody designed that form, and when they did, they made assumptions about what everyone's information looks like. Most of the time nobody even notices the assumptions, until somebody runs into one.

In Lesson 5 we saw Patrick McKenzie's famous list, "Falsehoods Programmers Believe About Names." This week, you're going to make that kind of list yourself, for each other's data.

Here's the game. You'll design a small **schema** (the set of fields, and the rules for what goes in each field) for something you know well, and write a little checker that enforces it. Then you'll post it, and your classmates will try to **break it**: find a real thing, or at least a realistic one, that your schema can't hold honestly. And you'll do the same to theirs. The goal isn't to make a perfect schema, because there's no such thing! The goal is to find the assumptions hiding inside every schema, including the ones the author didn't know they were making.

> **Running this:** Click a gray code cell and press **Shift+Enter** or its play button. Run the cells from top to bottom. If a later cell says a name isn't defined, go back and run the cells above it first.

## An example schema: a school enrollment form

Here's a real kind of form that shows up all the time, on school enrollment paperwork, at doctors' offices, and in all sorts of systems that keep records about kids. Each child gets a record, a dictionary like the ones from Lesson 4, and the form's schema looks like this:

| Field | What it means | The rule |
|---|---|---|
| `child_name` | the child's name | required; some text |
| `grade` | which grade they're starting | must be one of: K, 1, 2, … 12 |
| `mother_name` | the child's mother | required; some text |
| `father_name` | the child's father | required; some text |

That seems pretty reasonable, right? Here's the schema written as Python, along with a checker function that goes through a record and lists every rule it breaks:

In [ ]:
ALLOWED_GRADES = ["K", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11", "12"]
REQUIRED_FIELDS = ["child_name", "grade", "mother_name", "father_name"]


def check_child(record):
    problems = []

    for field in REQUIRED_FIELDS:
        if field not in record or record[field] == "":
            problems.append(f"missing '{field}'")

    if "grade" in record and record["grade"] not in ALLOWED_GRADES:
        problems.append(f"'{record['grade']}' isn't an allowed grade")

    return problems


def report(record):
    problems = check_child(record)
    if len(problems) == 0:
        print(f"{record['child_name']}: fits the schema")
    else:
        print(f"{record['child_name']}: DOESN'T FIT")
        for problem in problems:
            print("   -", problem)

Everything in there is something you've seen before: a list of allowed values, a loop over the required fields, `in` to check whether a key exists, and an `if` for each rule.

Let's try it on one child:

In [ ]:
report({"child_name": "Maya", "grade": "3", "mother_name": "Ana", "father_name": "Luis"})

Maya fits. Great! Done, right?

## Now let's break it

Here are some kids who are completely real, and whom this form can't handle:

In [ ]:
report({"child_name": "Dev", "grade": "5", "guardian": "his older sister, Priya"})
report({"child_name": "Sam", "grade": "2", "mother_name": "Jo", "father_name": ""})
report({"child_name": "Lily", "grade": "TK", "mother_name": "Mei", "father_name": "Tom"})

Each one breaks a different assumption:

- **Dev** is being raised by his older sister, so there's no parent to write in either box. The schema assumed every child is being raised by their parents.
- **Sam's** father died when Sam was a baby, or Sam has never known who he is. The box is required, so the form won't let Sam's mom leave it blank. Remember from Lesson 5 how "unknown" is different from "nothing"? If she types "none" or "unknown" just to get past the checker, the record now has something in the father box that isn't a name at all.
- **Lily** is in transitional kindergarten, which a lot of schools have. The list of grades was really "the grades the form's designer thought of," and it got treated as if it were "all the grades."

And here's a sneakier kind of break, where the record *passes* the checker but the data still ends up wrong:

In [ ]:
report({"child_name": "Noor", "grade": "1", "mother_name": "Rana", "father_name": "Beth"})

That one "fits." But Noor has two moms, so one of them had to be written in the "father" box. The checker has no idea anything's wrong, and now the school's records say something that isn't true. The same thing happens to kids being raised by grandparents, an aunt, or foster parents: either the form has no place for the people actually taking care of them, or somebody gets squeezed into a box that doesn't describe them.

The schema assumed that every child has exactly one mother and one father, that both of them are known, and that they're the people actually raising the child. Lots of families don't look like that! And on a form like this, the boxes aren't just labels. They can decide who's allowed to pick a kid up from school, or who gets the phone call in an emergency. So when the schema doesn't fit, it's a real kid's real family that gets left out.

So what should the form do instead? There's no single perfect answer, but a better design doesn't try to guess what a family looks like. It lets a record have any number of guardians, and lets each one say how they're related to the child:

In [ ]:
dev = {
    "child_name": "Dev",
    "grade": "5",
    "guardians": [
        {"name": "Priya", "relationship": "older sister"},
    ],
}

for guardian in dev["guardians"]:
    print(f"{guardian['name']} ({guardian['relationship']})")

That's a list of dictionaries inside a dictionary, all things you've seen in this unit. It's a little more complicated than two boxes, but it can hold Dev's family, and Sam's, and Noor's, without anybody having to lie to the form. (And Lily's problem has an even simpler fix: add "TK" to the list, or let the school type in its own grade names.) Every fix is a choice, though, and every choice leaves *somebody* in or out, which is exactly what Lesson 5 was about.

## Places assumptions like to hide

When you're hunting for breaks in a classmate's schema, here are some good questions to ask:

- **"Exactly one."** Does the schema assume every item has exactly one of something? One name, one category, one owner, one mother and one father?
- **"Always known."** Does it require a value that someone might honestly not know?
- **"This list covers everything."** Is there a list of allowed choices? What's missing from it?
- **"Never changes."** Could the value change over time, like a name, an address, or a size?
- **"Whole numbers only," "always positive," "never zero."** What are the rules on the numbers, and what real case breaks them?
- **"One record, one thing."** Could one real thing need several records, or could several real things get squashed into one?

## Your turn: build a schema worth breaking

Pick something you know a lot about: a collection, a hobby, a game, the food in your fridge, the plants on your windowsill, the bands you've seen live, &c. The enrollment form was a serious example on purpose, but for your own schema, keep it low-stakes and safe to share. Don't use real people's private information, because your schema is going to be public on the discussion board!

Then, in the cells below:

1. Write your schema as a table, like the enrollment one: each field, what it means, and its rule. Aim for **four or five fields**, and include at least one list of allowed choices and at least one rule about a number (like "a whole number, 0 or more").
2. Write a checker function for it. You can copy `check_child` and change it to fit your fields. For a number rule, `type(value) != int` (remember `type()` from Unit 3?) asks "is this something other than a whole number?", and a comparison like `value < 0` checks the range.
3. Make three records that fit, and run the checker on them.
4. Before you post, try to break it yourself! Find one case your schema can't handle, and decide whether to fix it or leave it.

| Field | What it means | The rule |
|---|---|---|
| | | |
| | | |
| | | |
| | | |

In [ ]:
# YOUR SCHEMA AND CHECKER GO HERE.

# 1. Lists of allowed values and required fields.

# 2. A check_ function that returns a list of problems.

# 3. A few records that fit, and a loop that reports on each one.

## Reflection

A sentence or two each, after your classmates have had a go at your schema:

1. What was the most surprising break someone found in your schema? What assumption had you made without realizing it?
2. What did you decide to do about it: change the schema, or keep it the way it was? What would the change cost, and who would be left out if you didn't make it?
3. Think of a real form or system you've used that made an assumption like these. Who might it fail?

## What to post

Post on the **Week 5 discussion thread**:

1. your schema table;
2. your checker code and the output from your three records (pasted as text); and
3. one break you found in your own schema, and what you decided to do about it.

Then, the fun part: **try to break at least two classmates' schemas.** For each one, describe a real or realistic item that either gets rejected even though it's legitimate, or gets accepted but recorded wrong (like Noor's record). Name the assumption it breaks. The best replies find something the author never saw coming!

When people break yours, reply and say what you'd change, or why you'd keep it the way it is. Then come back and answer the reflection questions.